In [3]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# AlphaFold 3: Protein-Ligand Docking Prediction Example
<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphafold3/cloudai_alphafold3_vai_quickstart.ipynb">
      <img src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fcommunity%2Falphafold3%2Fcloudai_alphafold3_vai_quickstart.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphafold3/cloudai_alphafold3_vai_quickstart.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Agent Platform Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/alphafold3/cloudai_alphafold3_vai_quickstart.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

## Overview

AlphaFold 3 predicts the 3D structure of proteins, DNA, RNA, chemical modifications, and small-molecule ligands. It enables researchers to simulate physical interactions directly from sequence inputs and chemical descriptions.

This tutorial covers an end-to-end prediction workflow for AlphaFold 3 deployed on Model Garden on the Google Cloud Gemini Enterprise Agent Platform.

This tutorial covers:
1. Authenticating and connecting to the Google Cloud environment.
2. Submitting a prediction request. The example shows a covalent complex of the **KRAS G12C mutant protein** bound to the inhibitor **Sotorasib (AMG-510)**.
3. Understanding Output Structure & Artifacts.
4. Comparing & Filtering Candidate Samples.
5. Extracting & Rendering 3D Coordinates.

## Getting Started

### Install Dependencies
First, install the libraries required to download files from GCS, manipulate matrices, and visualize 3D molecular coordinates.

In [4]:
%pip install -q py3Dmol biopython matplotlib numpy google-cloud-storage google-cloud-aiplatform

### Authenticate the Notebook Environment (Colab Only)
If running this notebook in Google Colab, authenticate the account to grant access to Google Cloud resources.

In [5]:
# Import required libraries
import json
import os
import sys

import google.auth
import google.auth.transport.requests
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
import requests
from google.cloud import aiplatform, storage

# Authenticate environment for Google Colab only
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

## Prerequisite: Deploy AlphaFold 3 on Model Garden

AlphaFold 3 is available as a self-deployed model on Model Garden on Gemini Enterprise Agent Platform. Users can deploy the model into their own secure Google Cloud environments. Refer to the official documentation for more details on deploying and scaling the model.

**Note:** The next steps in this notebook assume an AlphaFold 3 model is deployed and actively running on a dedicated endpoint.

### Configure Project Variables
Set the Google Cloud Project ID, the region where the AlphaFold 3 endpoint is deployed, and the GCS bucket where inference outputs will be saved.

In [6]:
PROJECT_ID = ""  # @param {type:"string"}
LOCATION = ""  # @param {type:"string"}
ENDPOINT_ID = ""  # @param {type:"string"}
BUCKET_NAME = ""  # @param {type:"string"}
JOB_NAME = ""  # @param {type:"string"}

## Step 1: Submit the Prediction Request

Configure the AlphaFold 3 request JSON payload. The example showcases how to predict interactions for two molecules:
1. **KRAS G12C Protein Chain (Entity A)**: A 189 amino acid mutant protein.
2. **Sotorasib Ligand (Entity B)**: Defined by its CCD code `MOV`.

To model the covalent interaction, explicitly define the covalent bond in the `bondedAtomPairs` list. Specify that the Sulfur-Gamma (`SG`) atom of residue 12 (Cysteine) on the protein chain (`A`) forms a covalent bond with the Carbon 25 (`C25`) atom of the Sotorasib ligand (`B`).

**Note on Output Handling:** While the response can be returned directly through the API, it is highly recommended to provide a GCS bucket (`output_dir`) to persist the output. This also enables capturing a more detailed output dataset.

In [ ]:
# --- Define AlphaFold 3 Inputs ---
SEEDS = [42, 43]

PROTEIN_ID = "A"
PROTEIN_SEQ = "MTEYKLVVVGACGVGKSALTIQLIQNHFVDEYDPTIEDSYRKQVVIDGETCLLDILDTAGQEEYSAMRDQYMRTGEGFLCVFAINNTKSFEDIHHYREQIKRVKDSEDVPMVLVGNKCDLPSRTVDTKQAQDLARSYGIPFIETSAKTRQRVEDAFYTLVREIRQYRLKKISKEEKTPGCVKIKKCIIM"
PROTEIN_DESC = "Human KRAS4B G12C mutant (UniProt P01116 with G12C, 189 aa)"

LIGAND_ID = "B"
LIGAND_CCD = "MOV"
LIGAND_DESC = "Sotorasib / AMG-510 (CCD: MOV), covalent KRAS G12C inhibitor"

# Define covalent bond pair: [atom_1, atom_2] where atom = [entity_id, residue_index, atom_name]
BONDED_ATOM_PAIRS = [[["A", 12, "SG"], ["B", 1, "C25"]]]

OUTPUT_DIR = f"gs://{BUCKET_NAME}/my_dir"

# --- Construct AlphaFold 3 Prediction Request Payload ---
# --- Overwrites the output directory, if one exists already ---
payload = {
    "instances": [
        {
            "name": JOB_NAME,
            "dialect": "alphafold3",
            "version": 4,
            "modelSeeds": SEEDS,
            "sequences": [
                {
                    "protein": {
                        "id": PROTEIN_ID,
                        "sequence": PROTEIN_SEQ,
                        "description": PROTEIN_DESC,
                    }
                },
                {
                    "ligand": {
                        "id": LIGAND_ID,
                        "ccdCodes": [LIGAND_CCD],
                        "description": LIGAND_DESC,
                    }
                },
            ],
            "bondedAtomPairs": BONDED_ATOM_PAIRS,
        }
    ],
    "parameters": {
        "output_dir": OUTPUT_DIR,
        "force_output_dir": True,
        "run_data_pipeline": True,
    },
}

# --- Submit Prediction Request ---
aiplatform.init(project=PROJECT_ID, location=LOCATION)
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

response = endpoint.predict(
    instances=payload["instances"], parameters=payload["parameters"]
)

print("Prediction Request Submitted Successfully:")
print(response)

## Step 2: Inline Response vs. Saved Artifacts

AlphaFold 3 provides two delivery mechanisms for retrieving structural prediction outputs. By default, the API response returns prediction results inline over HTTP. For production workloads, specifying a Google Cloud Storage (GCS) bucket exports the complete multi-sample artifact dataset.

In [8]:
def download_af3_outputs(
    bucket_name, job_name, seed=None, sample_idx=None, local_dir="."
):
    """Downloads CIF coordinates, confidences, and summary files from GCS.

    If seed and sample_idx are None, downloads the overall #1 top-ranked candidate
    model directly from the top-level job directory.
    """
    if bucket_name == "your-bucket-name":
        print("Using local sample data files for demonstration.")
        print(
            "Note: Update BUCKET_NAME with your Google Cloud Storage bucket to download live prediction outputs."
        )
        return

    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(bucket_name)

    if seed is not None and sample_idx is not None:
        gcs_prefix = f"my_dir/{job_name}/seed-{seed}_sample-{sample_idx}"
        file_prefix = f"{job_name}_seed-{seed}_sample-{sample_idx}"
    else:
        gcs_prefix = f"my_dir/{job_name}"
        file_prefix = job_name

    cif_blob = bucket.blob(f"{gcs_prefix}/{file_prefix}_model.cif")
    cif_blob.download_to_filename("model.cif")

    conf_blob = bucket.blob(f"{gcs_prefix}/{file_prefix}_confidences.json")
    conf_blob.download_to_filename("confidence.json")

    summary_blob = bucket.blob(f"{gcs_prefix}/{file_prefix}_summary_confidences.json")
    summary_blob.download_to_filename("summary.json")

    # Download ranking scores CSV if present
    try:
        ranking_blob = bucket.blob(f"my_dir/{job_name}/{job_name}_ranking_scores.csv")
        ranking_blob.download_to_filename("ranking_scores.csv")
    except Exception:
        pass

    print(
        "Downloaded top-ranked model.cif, confidence.json, and summary.json successfully."
    )


# Run the download function for the prediction outputs (fetches overall top-ranked structure)
download_af3_outputs(BUCKET_NAME, JOB_NAME)

Downloaded top-ranked model.cif, confidence.json, and summary.json successfully.


## Step 3: Comparing Candidate Samples

Because AlphaFold 3 utilizes a generative diffusion module, each run produces an ensemble of candidate structures across random seeds. Inspect `ranking_scores.csv` and `summary.json` **first** to identify the top-ranked candidate and verify that the model is free of steric clashes (`has_clash == 0`).

In [9]:
import csv
import json

print("=== Candidate Samples Ledger (ranking_scores.csv) ===")
with open("ranking_scores.csv", newline="", encoding="utf-8") as f:
    rows = sorted(
        csv.DictReader(f), key=lambda x: float(x["ranking_score"]), reverse=True
    )

# Calculate column widths cleanly and readably
headers = list(rows[0].keys())
widths = {}
for col in headers:
    lengths = [len(col)] + [len(r[col]) for r in rows]
    widths[col] = max(lengths)

print("  ".join(col.rjust(widths[col]) for col in headers))
for r in rows:
    print("  ".join(r[col].rjust(widths[col]) for col in headers))

top = rows[0]
print(
    f"\nPromoted Top Candidate: Seed {int(top['seed'])}, Sample"
    f" {int(top['sample'])} (Score: {float(top['ranking_score']):.4f})"
)

print("\n=== Top Candidate Quality Validation (summary.json) ===")
with open("summary.json", encoding="utf-8") as f:
    summary = json.load(f)

clash_str = "DETECTED (FAIL)" if summary.get("has_clash") else "None Detected (PASS)"
print("Top Candidate Metrics:")
print(f"  • Ranking Score : {summary.get('ranking_score', 'N/A')}")
print(f"  • Global pTM    : {summary.get('ptm', 'N/A')}")
print(f"  • Interface ipTM: {summary.get('iptm', 'N/A')}")
print(f"  • Steric Clash  : {clash_str}")

=== Candidate Samples Ledger (ranking_scores.csv) ===
seed  sample       ranking_score
  42       0  0.9765861758441248
  43       3  0.9737541527867797
  43       4  0.9733747951235855
  42       2  0.9732408715975229
  42       3  0.9723226792181007
  42       1   0.970259597191794
  43       0   0.968876439745061
  43       2  0.9686427701415588
  43       1  0.9618608651497567
  42       4   0.956112724247255

Promoted Top Candidate: Seed 42, Sample 0 (Score: 0.9766)

=== Top Candidate Quality Validation (summary.json) ===
Top Candidate Metrics:
  • Ranking Score : 0.98
  • Global pTM    : 0.85
  • Interface ipTM: 0.95
  • Steric Clash  : None Detected (PASS)


## Step 4: Inspecting 3D Structures

When reviewing the downloaded atomic structure `model.cif` for the top-ranked candidate, note key structural conventions: heavy-atom representation ($x,y,z$) and per-atom local confidence (pLDDT) stored in the B-factor column (`_atom_site.B_iso_or_equiv`).

In [10]:
import py3Dmol
from Bio.PDB import MMCIFParser
from IPython.display import display

with open("model.cif") as f:
    cif_data = f.read()

# Map AlphaFold pLDDT confidence color bands
color_map = {
    "#ff7d45": [],  # Very Low (<50)
    "#ffdb13": [],  # Low (50-70)
    "#65cbf3": [],  # Confident (70-90)
    "#0053d6": [],  # Very High (>=90)
}

# Parse mmCIF file using BioPython
parser = MMCIFParser(QUIET=True)
structure = parser.get_structure("af3", "model.cif")

# Extract atom serial numbers based on B-factor (pLDDT)
for atom in structure.get_atoms():
    serial = atom.get_serial_number()
    b_factor = atom.bfactor

    if b_factor < 50:
        color_map["#ff7d45"].append(serial)
    elif b_factor < 70:
        color_map["#ffdb13"].append(serial)
    elif b_factor < 90:
        color_map["#65cbf3"].append(serial)
    else:
        color_map["#0053d6"].append(serial)

# Initialize py3Dmol viewer
view = py3Dmol.view(width=800, height=500)
view.addModel(cif_data, "cif")

# Apply pLDDT cartoon styling to polymer backbones
view.setStyle({"model": -1}, {})
for color, atoms in color_map.items():
    if atoms:
        view.addStyle({"serial": atoms}, {"cartoon": {"color": color}})

# Render covalent ligand Sotorasib (CCD: MOV) explicitly as green-carbon sticks
view.addStyle({"hetflag": True}, {"stick": {"colorscheme": "greenCarbon"}})

view.zoomTo()
view.spin(True)
display(view.show())

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

None